# Environment

In [1200]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/iraklizerekidze/house-prices/sample_submission.csv
/kaggle/input/datasets/iraklizerekidze/house-prices/data_description.txt
/kaggle/input/datasets/iraklizerekidze/house-prices/train.csv
/kaggle/input/datasets/iraklizerekidze/house-prices/test.csv


In [1201]:
!pip install mlflow dagshub

In [1202]:
import dagshub
dagshub.init(repo_owner='izere23', repo_name='Assignment-1---Advanced-Regression', mlflow=True)

Initialized MLflow to track repo "izere23/Assignment-1---Advanced-Regression"

Repository izere23/Assignment-1---Advanced-Regression initialized!

In [1203]:
import mlflow
mlflow.sklearn.autolog(disable=True)


# Imports

In [1204]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, mean_squared_log_error

import mlflow
import mlflow.sklearn

# Splitting

In [1205]:
df = pd.read_csv('/kaggle/input/datasets/iraklizerekidze/house-prices/train.csv')

X_train, X_test = train_test_split(df, test_size=0.2, random_state=42)

Y_train = X_train.pop('SalePrice')
Y_test = X_test.pop('SalePrice')

X_train_no_id = X_train.drop(columns='Id')
X_test_no_id = X_test.drop(columns='Id')

print(X_train_no_id.shape, X_test_no_id.shape)

(1168, 79) (292, 79)


# Cleaning

## High Nan Columns

In [1206]:
missing_percent = X_train_no_id.isnull().mean() * 100
missing_percent = missing_percent.sort_values(ascending=False)

high_nan_threshold = 80
high_nan_cols = missing_percent[missing_percent > high_nan_threshold].index.tolist()

X_train_no_nan = X_train_no_id.drop(columns=high_nan_cols)
X_test_no_nan = X_test_no_id.drop(columns=high_nan_cols)

print("Dropped high-NaN columns:", len(high_nan_cols))
print(X_train_no_nan.shape, X_test_no_nan.shape)

Dropped high-NaN columns: 4
(1168, 75) (292, 75)


## High Dominance Columns

In [1207]:
dominance_threshold = 95
dominance = []

for col in X_train_no_nan.columns:
    top_freq = X_train_no_nan[col].value_counts(normalize=True, dropna=False).iloc[0]
    dominance.append(top_freq)

dominance_series = pd.Series(dominance, index=X_train_no_nan.columns).sort_values(ascending=False)

low_info_cols = dominance_series[dominance_series > dominance_threshold / 100].index.tolist()

X_train_cln = X_train_no_nan.drop(columns=low_info_cols)
X_test_cln = X_test_no_nan.drop(columns=low_info_cols)

print("Dropped low-info columns:", len(low_info_cols))
print(X_train_cln.shape, X_test_cln.shape)

Dropped low-info columns: 10
(1168, 65) (292, 65)


In [1208]:
num_cols = X_train_cln.select_dtypes(include=['number']).columns.tolist()
cat_cols = X_train_cln.select_dtypes(exclude=['number']).columns.tolist()

print("Numerical columns:", len(num_cols))
print("Categorical columns:", len(cat_cols))

Numerical columns: 31
Categorical columns: 34


In [1209]:
median_values = X_train_cln[num_cols].median()
X_train_cln[num_cols] = X_train_cln[num_cols].fillna(median_values)
X_test_cln[num_cols] = X_test_cln[num_cols].fillna(median_values)

mode_values = X_train_cln[cat_cols].mode().iloc[0]
X_train_cln[cat_cols] = X_train_cln[cat_cols].fillna(mode_values)
X_test_cln[cat_cols] = X_test_cln[cat_cols].fillna(mode_values)

print("Remaining missing values in train:", X_train_cln.isnull().sum().sum())
print("Remaining missing values in test:", X_test_cln.isnull().sum().sum())

Remaining missing values in train: 0
Remaining missing values in test: 0


# Feature Engineering
## Total Bathrooms

In [1210]:
X_train_cln['TotalBath'] = (
    X_train_cln['FullBath'] +
    0.5 * X_train_cln['HalfBath'] +
    X_train_cln['BsmtFullBath'] +
    0.5 * X_train_cln['BsmtHalfBath']
)

X_test_cln['TotalBath'] = (
    X_test_cln['FullBath'] +
    0.5 * X_test_cln['HalfBath'] +
    X_test_cln['BsmtFullBath'] +
    0.5 * X_test_cln['BsmtHalfBath']
)

X_train_cln['OverallScore'] = X_train_cln['OverallQual'] * X_train_cln['OverallCond']
X_test_cln['OverallScore'] = X_test_cln['OverallQual'] * X_test_cln['OverallCond']

X_train_cln['QualArea'] = X_train_cln['OverallScore'] * X_train_cln['GrLivArea']
X_test_cln['QualArea'] = X_test_cln['OverallScore'] * X_test_cln['GrLivArea']

X_train_cln['LogGrLivArea'] = np.log1p(X_train_cln['GrLivArea'])
X_test_cln['LogGrLivArea'] = np.log1p(X_test_cln['GrLivArea'])

X_train_cln['HouseAge'] = X_train_cln['YrSold'] - X_train_cln['YearBuilt']
X_test_cln['HouseAge'] = X_test_cln['YrSold'] - X_test_cln['YearBuilt']

X_train_cln['HouseAge_sq'] = X_train_cln['HouseAge'] ** 2
X_test_cln['HouseAge_sq'] = X_test_cln['HouseAge'] ** 2

print(X_train_cln.shape, X_test_cln.shape)

(1168, 71) (292, 71)


In [1211]:
X_train_encoded = pd.get_dummies(X_train_cln, columns=cat_cols, drop_first=False)
X_test_encoded = pd.get_dummies(X_test_cln, columns=cat_cols, drop_first=False)

X_train_encoded, X_test_encoded = X_train_encoded.align(X_test_encoded, join='left', axis=1, fill_value=0)

print("Train encoded shape:", X_train_encoded.shape)
print("Test encoded shape:", X_test_encoded.shape)

Train encoded shape: (1168, 248)
Test encoded shape: (292, 248)


# One Hot Encoding

In [1212]:
X_train_encoded = pd.get_dummies(X_train_cln, columns=cat_cols, drop_first=False)
X_test_encoded = pd.get_dummies(X_test_cln, columns=cat_cols, drop_first=False)

X_train_encoded, X_test_encoded = X_train_encoded.align(X_test_encoded, join='left', axis=1, fill_value=0)

print("Train encoded shape:", X_train_encoded.shape)
print("Test encoded shape:", X_test_encoded.shape)

Train encoded shape: (1168, 248)
Test encoded shape: (292, 248)


# Feature Selection

## Correlation with target

In [1213]:
df_corr = pd.concat([X_train_encoded, Y_train], axis=1)

corr_matrix = df_corr.corr(numeric_only=True)
target_corr = corr_matrix['SalePrice'].abs().sort_values(ascending=False)

low_corr_features = target_corr[target_corr < 0.05].index.tolist()
low_corr_features = [f for f in low_corr_features if f != 'SalePrice']

X_train_cor = X_train_encoded.drop(columns=low_corr_features, errors='ignore')
X_test_cor = X_test_encoded.drop(columns=low_corr_features, errors='ignore')

print("Dropped low-correlation features:", len(low_corr_features))
print(X_train_cor.shape, X_test_cor.shape)

Dropped low-correlation features: 72
(1168, 176) (292, 176)


## Multicollinearity filter

In [1214]:
corr_matrix_features = X_train_cor.corr().abs()

upper = corr_matrix_features.where(
    np.triu(np.ones(corr_matrix_features.shape), k=1).astype(bool)
)

to_drop = [col for col in upper.columns if any(upper[col] > 0.9)]

X_train_mul = X_train_cor.drop(columns=to_drop, errors='ignore')
X_test_mul = X_test_cor.drop(columns=to_drop, errors='ignore')

print("Dropped multicollinear features:", len(to_drop))
print(X_train_mul.shape, X_test_mul.shape)

Dropped multicollinear features: 12
(1168, 164) (292, 164)


# Training

## Linear Regression

In [1215]:
with mlflow.start_run(run_name="LinearRegression_Final"):
    
    X_train_final = X_train_mul.copy()
    X_test_final = X_test_mul.copy()

    mlflow.log_param("phase", "Feature_Selection")
    mlflow.log_param("features", "all")
    mlflow.log_param("imputation", "Median + Mode")
    mlflow.log_param("scaling", "StandardScaler")
    mlflow.log_param("model", "LinearRegression")
    mlflow.log_param("num_features", X_train_final.shape[1])
    mlflow.log_param("dominance_threshold", 0.95)
    mlflow.log_param("high_nan_threshold", 0.80)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_final)
    X_test_scaled = scaler.transform(X_test_final)

    model_lr = LinearRegression()
    model_lr.fit(X_train_scaled, Y_train)

    train_preds = model_lr.predict(X_train_scaled)
    test_preds = model_lr.predict(X_test_scaled)

    train_r2 = r2_score(Y_train, train_preds)
    test_r2 = r2_score(Y_test, test_preds)

    train_rmse = np.sqrt(mean_squared_error(Y_train, train_preds))
    test_rmse = np.sqrt(mean_squared_error(Y_test, test_preds))

    train_mae = mean_absolute_error(Y_train, train_preds)
    test_mae = mean_absolute_error(Y_test, test_preds)

    train_rmsle = np.sqrt(mean_squared_log_error(Y_train, np.maximum(0, train_preds)))
    test_rmsle = np.sqrt(mean_squared_log_error(Y_test, np.maximum(0, test_preds)))

    overfit_gap = train_r2 - test_r2

    mlflow.log_metric("train_r2", train_r2)
    mlflow.log_metric("test_r2", test_r2)
    mlflow.log_metric("train_rmse", train_rmse)
    mlflow.log_metric("test_rmse", test_rmse)
    mlflow.log_metric("train_mae", train_mae)
    mlflow.log_metric("test_mae", test_mae)
    mlflow.log_metric("train_rmsle", train_rmsle)
    mlflow.log_metric("test_rmsle", test_rmsle)
    mlflow.log_metric("overfitting_gap", overfit_gap)

    mlflow.sklearn.log_model(
        sk_model=model_lr,
        artifact_path="linear_regression_model"
    )

    print(f"Train R²: {train_r2:.4f}")
    print(f"Test R²: {test_r2:.4f}")
    print(f"Train RMSE: {train_rmse:.2f}")
    print(f"Test RMSE: {test_rmse:.2f}")
    print(f"Train MAE: {train_mae:.2f}")
    print(f"Test MAE: {test_mae:.2f}")
    print(f"Train RMSLE: {train_rmsle:.4f}")
    print(f"Test RMSLE: {test_rmsle:.4f}")
    print(f"Overfitting Gap: {overfit_gap:.4f}")

2026/04/13 18:22:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/13 18:23:08 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Train R²: 0.8953
Test R²: 0.8789
Train RMSE: 24985.91
Test RMSE: 30478.01
Train MAE: 15779.28
Test MAE: 20134.12
Train RMSLE: 0.1263
Test RMSLE: 0.1557
Overfitting Gap: 0.0164
🏃 View run LinearRegression_Final at: https://dagshub.com/izere23/Assignment-1---Advanced-Regression.mlflow/#/experiments/2/runs/b176200e245a40389435e1802bb72bb2
🧪 View experiment at: https://dagshub.com/izere23/Assignment-1---Advanced-Regression.mlflow/#/experiments/2


## Linear Regression With Log Target

In [1216]:
import mlflow
import mlflow.sklearn
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, mean_squared_log_error

with mlflow.start_run(run_name="LinearRegression_Log_Final"):

    X_train_final = X_train_encoded.copy()
    X_test_final = X_test_encoded.copy()

    y_train_log = np.log1p(Y_train)
    y_test_log = np.log1p(Y_test)

    mlflow.log_param("model", "LinearRegression")
    mlflow.log_param("target_transform", "log1p")
    mlflow.log_param("engineered_features", True)
    mlflow.log_param("feature_selection", False)
    mlflow.log_param("imputation_strategy", "median/mode")
    mlflow.log_param("scaling", "StandardScaler")
    mlflow.log_param("num_features", X_train_final.shape[1])

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_final)
    X_test_scaled = scaler.transform(X_test_final)

    model_lr = LinearRegression()
    model_lr.fit(X_train_scaled, y_train_log)

    train_preds_log = model_lr.predict(X_train_scaled)
    test_preds_log = model_lr.predict(X_test_scaled)

    train_preds = np.expm1(train_preds_log)
    test_preds = np.expm1(test_preds_log)

    train_preds = np.maximum(0, train_preds)
    test_preds = np.maximum(0, test_preds)

    train_r2 = r2_score(Y_train, train_preds)
    test_r2 = r2_score(Y_test, test_preds)

    train_rmse = np.sqrt(mean_squared_error(Y_train, train_preds))
    test_rmse = np.sqrt(mean_squared_error(Y_test, test_preds))

    train_mae = mean_absolute_error(Y_train, train_preds)
    test_mae = mean_absolute_error(Y_test, test_preds)

    train_rmsle = np.sqrt(mean_squared_log_error(Y_train, train_preds))
    test_rmsle = np.sqrt(mean_squared_log_error(Y_test, test_preds))

    overfit_gap = train_r2 - test_r2

    mlflow.log_metric("train_r2", train_r2)
    mlflow.log_metric("test_r2", test_r2)
    mlflow.log_metric("train_rmse", train_rmse)
    mlflow.log_metric("test_rmse", test_rmse)
    mlflow.log_metric("train_mae", train_mae)
    mlflow.log_metric("test_mae", test_mae)
    mlflow.log_metric("train_rmsle", train_rmsle)
    mlflow.log_metric("test_rmsle", test_rmsle)
    mlflow.log_metric("overfitting_gap", overfit_gap)

    mlflow.sklearn.log_model(
        sk_model=model_lr,
        artifact_path="linear_regression_log_model"
    )

    print(f"Train R²: {train_r2:.4f}")
    print(f"Test R²: {test_r2:.4f}")
    print(f"Train RMSE: {train_rmse:.2f}")
    print(f"Test RMSE: {test_rmse:.2f}")
    print(f"Train MAE: {train_mae:.2f}")
    print(f"Test MAE: {test_mae:.2f}")
    print(f"Train RMSLE: {train_rmsle:.4f}")
    print(f"Test RMSLE: {test_rmsle:.4f}")
    print(f"Overfitting Gap: {overfit_gap:.4f}")

2026/04/13 18:23:34 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/13 18:23:48 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Train R²: 0.9220
Test R²: 0.8976
Train RMSE: 21574.78
Test RMSE: 28021.89
Train MAE: 12905.31
Test MAE: 16994.14
Train RMSLE: 0.1004
Test RMSLE: 0.1316
Overfitting Gap: 0.0243
🏃 View run LinearRegression_Log_Final at: https://dagshub.com/izere23/Assignment-1---Advanced-Regression.mlflow/#/experiments/2/runs/6ca0e5e927ee4530a0ee630cb8f2b39d
🧪 View experiment at: https://dagshub.com/izere23/Assignment-1---Advanced-Regression.mlflow/#/experiments/2


## Lasso with log target

In [1217]:
import mlflow
import mlflow.sklearn
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, mean_squared_log_error

with mlflow.start_run(run_name="Lasso_Log_Final"):

    X_train_final = X_train_encoded.copy()
    X_test_final = X_test_encoded.copy()

    y_train_log = np.log1p(Y_train)
    y_test_log = np.log1p(Y_test)

    mlflow.log_param("model", "Lasso")
    mlflow.log_param("alpha", 0.005)
    mlflow.log_param("target_transform", "log1p")
    mlflow.log_param("engineered_features", True)
    mlflow.log_param("feature_selection", False)
    mlflow.log_param("imputation_strategy", "median/mode")
    mlflow.log_param("scaling", "StandardScaler")
    mlflow.log_param("num_features", X_train_final.shape[1])

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_final)
    X_test_scaled = scaler.transform(X_test_final)

    model_lasso = Lasso(alpha=0.005, random_state=42, max_iter=10000)
    model_lasso.fit(X_train_scaled, y_train_log)

    train_preds_log = model_lasso.predict(X_train_scaled)
    test_preds_log = model_lasso.predict(X_test_scaled)

    train_preds = np.expm1(train_preds_log)
    test_preds = np.expm1(test_preds_log)

    train_preds = np.maximum(0, train_preds)
    test_preds = np.maximum(0, test_preds)

    train_r2 = r2_score(Y_train, train_preds)
    test_r2 = r2_score(Y_test, test_preds)

    train_rmse = np.sqrt(mean_squared_error(Y_train, train_preds))
    test_rmse = np.sqrt(mean_squared_error(Y_test, test_preds))

    train_mae = mean_absolute_error(Y_train, train_preds)
    test_mae = mean_absolute_error(Y_test, test_preds)

    train_rmsle = np.sqrt(mean_squared_log_error(Y_train, train_preds))
    test_rmsle = np.sqrt(mean_squared_log_error(Y_test, test_preds))

    overfit_gap = train_r2 - test_r2

    mlflow.log_metric("train_r2", train_r2)
    mlflow.log_metric("test_r2", test_r2)
    mlflow.log_metric("train_rmse", train_rmse)
    mlflow.log_metric("test_rmse", test_rmse)
    mlflow.log_metric("train_mae", train_mae)
    mlflow.log_metric("test_mae", test_mae)
    mlflow.log_metric("train_rmsle", train_rmsle)
    mlflow.log_metric("test_rmsle", test_rmsle)
    mlflow.log_metric("overfitting_gap", overfit_gap)

    mlflow.sklearn.log_model(
        sk_model=model_lasso,
        artifact_path="lasso_log_model"
    )

    print(f"Train R²: {train_r2:.4f}")
    print(f"Test R²: {test_r2:.4f}")
    print(f"Train RMSE: {train_rmse:.2f}")
    print(f"Test RMSE: {test_rmse:.2f}")
    print(f"Train MAE: {train_mae:.2f}")
    print(f"Test MAE: {test_mae:.2f}")
    print(f"Train RMSLE: {train_rmsle:.4f}")
    print(f"Test RMSLE: {test_rmsle:.4f}")
    print(f"Overfitting Gap: {overfit_gap:.4f}")

2026/04/13 18:24:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/13 18:24:29 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Train R²: 0.8886
Test R²: 0.9083
Train RMSE: 25772.54
Test RMSE: 26517.30
Train MAE: 14098.54
Test MAE: 15706.76
Train RMSLE: 0.1137
Test RMSLE: 0.1262
Overfitting Gap: -0.0197
🏃 View run Lasso_Log_Final at: https://dagshub.com/izere23/Assignment-1---Advanced-Regression.mlflow/#/experiments/2/runs/434b5aa44c4a4fe88a71e887585a692c
🧪 View experiment at: https://dagshub.com/izere23/Assignment-1---Advanced-Regression.mlflow/#/experiments/2


## Decision Tree

In [1218]:
import mlflow
import mlflow.sklearn
import numpy as np

from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, mean_squared_log_error

with mlflow.start_run(run_name="DecisionTree_Final"):

    X_train_final = X_train_encoded.copy()
    X_test_final = X_test_encoded.copy()

    mlflow.log_param("model", "DecisionTreeRegressor")
    mlflow.log_param("max_depth", 5)
    mlflow.log_param("min_samples_split", 10)
    mlflow.log_param("min_samples_leaf", 5)
    mlflow.log_param("engineered_features", True)
    mlflow.log_param("feature_selection", False)
    mlflow.log_param("imputation_strategy", "median/mode")
    mlflow.log_param("num_features", X_train_final.shape[1])

    model_dt = DecisionTreeRegressor(
        max_depth=5,
        min_samples_split=10,
        min_samples_leaf=5,
        random_state=42
    )

    model_dt.fit(X_train_final, Y_train)

    train_preds = model_dt.predict(X_train_final)
    test_preds = model_dt.predict(X_test_final)

    train_r2 = r2_score(Y_train, train_preds)
    test_r2 = r2_score(Y_test, test_preds)

    train_rmse = np.sqrt(mean_squared_error(Y_train, train_preds))
    test_rmse = np.sqrt(mean_squared_error(Y_test, test_preds))

    train_mae = mean_absolute_error(Y_train, train_preds)
    test_mae = mean_absolute_error(Y_test, test_preds)

    train_rmsle = np.sqrt(mean_squared_log_error(Y_train, np.maximum(0, train_preds)))
    test_rmsle = np.sqrt(mean_squared_log_error(Y_test, np.maximum(0, test_preds)))

    overfit_gap = train_r2 - test_r2

    mlflow.log_metric("train_r2", train_r2)
    mlflow.log_metric("test_r2", test_r2)
    mlflow.log_metric("train_rmse", train_rmse)
    mlflow.log_metric("test_rmse", test_rmse)
    mlflow.log_metric("train_mae", train_mae)
    mlflow.log_metric("test_mae", test_mae)
    mlflow.log_metric("train_rmsle", train_rmsle)
    mlflow.log_metric("test_rmsle", test_rmsle)
    mlflow.log_metric("overfitting_gap", overfit_gap)

    mlflow.sklearn.log_model(
        sk_model=model_dt,
        artifact_path="decision_tree_model"
    )

    print(f"Train R²: {train_r2:.4f}")
    print(f"Test R²: {test_r2:.4f}")
    print(f"Train RMSE: {train_rmse:.2f}")
    print(f"Test RMSE: {test_rmse:.2f}")
    print(f"Train MAE: {train_mae:.2f}")
    print(f"Test MAE: {test_mae:.2f}")
    print(f"Train RMSLE: {train_rmsle:.4f}")
    print(f"Test RMSLE: {test_rmsle:.4f}")
    print(f"Overfitting Gap: {overfit_gap:.4f}")

2026/04/13 18:24:56 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/13 18:25:10 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Train R²: 0.8595
Test R²: 0.7952
Train RMSE: 28945.60
Test RMSE: 39631.25
Train MAE: 19990.24
Test MAE: 25644.94
Train RMSLE: 0.1533
Test RMSLE: 0.1970
Overfitting Gap: 0.0643
🏃 View run DecisionTree_Final at: https://dagshub.com/izere23/Assignment-1---Advanced-Regression.mlflow/#/experiments/2/runs/f64d9707a09e4391ae2c600120066563
🧪 View experiment at: https://dagshub.com/izere23/Assignment-1---Advanced-Regression.mlflow/#/experiments/2


## Random Forest

In [1219]:
import mlflow
import mlflow.sklearn
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, mean_squared_log_error

with mlflow.start_run(run_name="RandomForest_Final"):

    X_train_final = X_train_encoded.copy()
    X_test_final = X_test_encoded.copy()

    mlflow.log_param("model", "RandomForestRegressor")
    mlflow.log_param("n_estimators", 1000)
    mlflow.log_param("max_depth", 12)
    mlflow.log_param("min_samples_leaf", 2)
    mlflow.log_param("engineered_features", True)
    mlflow.log_param("feature_selection", False)
    mlflow.log_param("imputation_strategy", "median/mode")
    mlflow.log_param("num_features", X_train_final.shape[1])

    model_rf = RandomForestRegressor(
        n_estimators=1000,
        max_depth=12,
        min_samples_leaf=2,
        max_features=0.7,
        random_state=42,
        n_jobs=-1
    )

    model_rf.fit(X_train_final, Y_train)

    train_preds = model_rf.predict(X_train_final)
    test_preds = model_rf.predict(X_test_final)

    train_r2 = r2_score(Y_train, train_preds)
    test_r2 = r2_score(Y_test, test_preds)

    train_rmse = np.sqrt(mean_squared_error(Y_train, train_preds))
    test_rmse = np.sqrt(mean_squared_error(Y_test, test_preds))

    train_mae = mean_absolute_error(Y_train, train_preds)
    test_mae = mean_absolute_error(Y_test, test_preds)

    train_rmsle = np.sqrt(mean_squared_log_error(Y_train, np.maximum(0, train_preds)))
    test_rmsle = np.sqrt(mean_squared_log_error(Y_test, np.maximum(0, test_preds)))

    overfit_gap = train_r2 - test_r2

    mlflow.log_metric("train_r2", train_r2)
    mlflow.log_metric("test_r2", test_r2)
    mlflow.log_metric("train_rmse", train_rmse)
    mlflow.log_metric("test_rmse", test_rmse)
    mlflow.log_metric("train_mae", train_mae)
    mlflow.log_metric("test_mae", test_mae)
    mlflow.log_metric("train_rmsle", train_rmsle)
    mlflow.log_metric("test_rmsle", test_rmsle)
    mlflow.log_metric("overfitting_gap", overfit_gap)

    mlflow.sklearn.log_model(
        sk_model=model_rf,
        artifact_path="random_forest_model"
    )

    print(f"Train R²: {train_r2:.4f}")
    print(f"Test R²: {test_r2:.4f}")
    print(f"Train RMSE: {train_rmse:.2f}")
    print(f"Test RMSE: {test_rmse:.2f}")
    print(f"Train MAE: {train_mae:.2f}")
    print(f"Test MAE: {test_mae:.2f}")
    print(f"Train RMSLE: {train_rmsle:.4f}")
    print(f"Test RMSLE: {test_rmsle:.4f}")
    print(f"Overfitting Gap: {overfit_gap:.4f}")

2026/04/13 18:25:40 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/13 18:25:51 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Train R²: 0.9710
Test R²: 0.8958
Train RMSE: 13151.33
Test RMSE: 28267.01
Train MAE: 7184.69
Test MAE: 16089.06
Train RMSLE: 0.0662
Test RMSLE: 0.1421
Overfitting Gap: 0.0752
🏃 View run RandomForest_Final at: https://dagshub.com/izere23/Assignment-1---Advanced-Regression.mlflow/#/experiments/2/runs/a90f713dfda644afb42b2c5edee8d9db
🧪 View experiment at: https://dagshub.com/izere23/Assignment-1---Advanced-Regression.mlflow/#/experiments/2


## XGBoost

In [1220]:
import mlflow
import mlflow.sklearn
import mlflow.xgboost
import numpy as np

from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, mean_squared_log_error

with mlflow.start_run(run_name="XGBoost_Final"):

    X_train_final = X_train_encoded.copy()
    X_test_final = X_test_encoded.copy()

    mlflow.log_param("model", "XGBRegressor")
    mlflow.log_param("n_estimators", 2000)
    mlflow.log_param("max_depth", 4)
    mlflow.log_param("learning_rate", 0.03)
    mlflow.log_param("subsample", 0.75)
    mlflow.log_param("colsample_bytree", 0.65)
    mlflow.log_param("engineered_features", True)
    mlflow.log_param("feature_selection", False)
    mlflow.log_param("imputation_strategy", "median/mode")
    mlflow.log_param("num_features", X_train_final.shape[1])

    model_xgb  = XGBRegressor(
        n_estimators=2000,
        learning_rate=0.03,
        max_depth=4,
        subsample=0.75,            
        colsample_bytree=0.65,     
        reg_alpha=0.2,             
        reg_lambda=1.5,            
        random_state=42,
        n_jobs=-1
    )


    model_xgb.fit(X_train_final, Y_train)

    train_preds = model_xgb.predict(X_train_final)
    test_preds = model_xgb.predict(X_test_final)

    train_r2 = r2_score(Y_train, train_preds)
    test_r2 = r2_score(Y_test, test_preds)

    train_rmse = np.sqrt(mean_squared_error(Y_train, train_preds))
    test_rmse = np.sqrt(mean_squared_error(Y_test, test_preds))

    train_mae = mean_absolute_error(Y_train, train_preds)
    test_mae = mean_absolute_error(Y_test, test_preds)

    train_rmsle = np.sqrt(mean_squared_log_error(Y_train, np.maximum(0, train_preds)))
    test_rmsle = np.sqrt(mean_squared_log_error(Y_test, np.maximum(0, test_preds)))

    overfit_gap = train_r2 - test_r2

    mlflow.log_metric("train_r2", train_r2)
    mlflow.log_metric("test_r2", test_r2)
    mlflow.log_metric("train_rmse", train_rmse)
    mlflow.log_metric("test_rmse", test_rmse)
    mlflow.log_metric("train_mae", train_mae)
    mlflow.log_metric("test_mae", test_mae)
    mlflow.log_metric("train_rmsle", train_rmsle)
    mlflow.log_metric("test_rmsle", test_rmsle)
    mlflow.log_metric("overfitting_gap", overfit_gap)

    mlflow.xgboost.log_model(
        xgb_model=model_xgb,
        artifact_path="xgboost_model"
    )

    print(f"Train R²: {train_r2:.4f}")
    print(f"Test R²: {test_r2:.4f}")
    print(f"Train RMSE: {train_rmse:.2f}")
    print(f"Test RMSE: {test_rmse:.2f}")
    print(f"Train MAE: {train_mae:.2f}")
    print(f"Test MAE: {test_mae:.2f}")
    print(f"Train RMSLE: {train_rmsle:.4f}")
    print(f"Test RMSLE: {test_rmsle:.4f}")
    print(f"Overfitting Gap: {overfit_gap:.4f}")

2026/04/13 18:26:20 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Train R²: 0.9994
Test R²: 0.9243
Train RMSE: 1853.17
Test RMSE: 24088.89
Train MAE: 1368.27
Test MAE: 14534.06
Train RMSLE: 0.0128
Test RMSLE: 0.1295
Overfitting Gap: 0.0751
🏃 View run XGBoost_Final at: https://dagshub.com/izere23/Assignment-1---Advanced-Regression.mlflow/#/experiments/2/runs/d4841afadc68451ea3e297e443b399cd
🧪 View experiment at: https://dagshub.com/izere23/Assignment-1---Advanced-Regression.mlflow/#/experiments/2


# MLFLOW

In [1221]:
import matplotlib.pyplot as plt

residuals = Y_test - Y_test_pred

fig1, ax1 = plt.subplots(figsize=(7, 4))
ax1.scatter(Y_test, Y_test_pred, alpha=0.5, color='royalblue')
ax1.plot([Y_test.min(), Y_test.max()], [Y_test.min(), Y_test.max()], 'r--', lw=2)
ax1.set_title("Actual vs Predicted Prices")
ax1.set_xlabel("Actual Prices")
ax1.set_ylabel("Predicted Prices")
plt.tight_layout()

fig2, ax2 = plt.subplots(figsize=(7, 4))
ax2.scatter(Y_test_pred, residuals, alpha=0.5, color='darkorange')
ax2.axhline(0, color='r', linestyle='--', lw=2)
ax2.set_title("Residuals (Errors) vs Predicted")
ax2.set_xlabel("Predicted Prices")
ax2.set_ylabel("Residuals (Actual - Predicted)")
plt.tight_layout()

fig3, ax3 = plt.subplots(figsize=(7, 4))
ax3.hist(residuals, bins=40, color='seagreen', edgecolor='black', alpha=0.7)
ax3.set_title("Distribution of Errors")
ax3.set_xlabel("Residual Value ($)")
ax3.set_ylabel("Frequency")
plt.tight_layout()


mlflow.set_experiment("House_Prices_Phase2")

with mlflow.start_run(run_name="FINAL EXPERIMENT"): 
    mlflow.log_param("phase", "Feature_Selection")
    mlflow.log_param("features", "all")
    mlflow.log_param("imputation", "Median + Mode")
    mlflow.log_param("scaling", "Standard")
    mlflow.log_param("model", "XGBoost") 
    mlflow.log_param("num_features", len(num_cols) + len(cat_cols))
    mlflow.log_param("Dominance Threshold", 0.95)
    mlflow.log_param("High Nan Threshold", 0.8)
    
    mlflow.log_metric("train_r2", train_r2)
    mlflow.log_metric("test_r2", test_r2)
    mlflow.log_metric("train_rmse", train_rmse)
    mlflow.log_metric("test_rmse", test_rmse)
    mlflow.log_metric("train_mae", train_mae)
    mlflow.log_metric("test_mae", test_mae)
    mlflow.log_metric("train_rmsle", train_rmsle)
    mlflow.log_metric("test_rmsle", test_rmsle)
    mlflow.log_metric("overfitting_gap", overfitting_gap)

    mlflow.log_figure(fig1, "plots/1_actual_vs_predicted.png")
    mlflow.log_figure(fig2, "plots/2_residuals.png")
    mlflow.log_figure(fig3, "plots/3_error_distribution.png")

    mlflow.sklearn.log_model(model, artifact_path="model")

plt.close(fig1)
plt.close(fig2)
plt.close(fig3)

print("Experiment logged with 3 separate visual graphs in MLflow!")

2026/04/13 18:27:06 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/13 18:27:20 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run FINAL EXPERIMENT at: https://dagshub.com/izere23/Assignment-1---Advanced-Regression.mlflow/#/experiments/2/runs/23bb4ebedec44e7f960023d6ee56be6a
🧪 View experiment at: https://dagshub.com/izere23/Assignment-1---Advanced-Regression.mlflow/#/experiments/2
Experiment logged with 3 separate visual graphs in MLflow!
